# 1. load package

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context


# 2. load and process data

In [ ]:
adata1 = sc.read_10x_h5("cell_feature_matrix_381.h5")
df1 = pd.read_csv("cells_381.csv.gz")

df1.set_index(adata1.obs_names, inplace=True)
adata1.obs = df1.copy()
adata1.obsm["spatial"] = adata1.obs[["x_centroid", "y_centroid"]].copy().to_numpy()


In [ ]:
adata2 = sc.read_10x_h5("cell_feature_matrix_488.h5")
df2 = pd.read_csv("cells_488.csv.gz")

df2.set_index(adata2.obs_names, inplace=True)
adata2.obs = df2.copy()
adata2.obsm["spatial"] = adata2.obs[["x_centroid", "y_centroid"]].copy().to_numpy()

In [ ]:
import anndata as ad
adata= ad.AnnData.concatenate(
    adata1,
    adata2,
    join='inner', 
    batch_key='batch',
    batch_categories=['381', '488'] 
 
)

In [ ]:
import pandas as pd
import anndata

roi_mapping = combined_df.set_index('Cell ID')['roi'].to_dict()


if 'roi' not in adata.obs.columns:
    adata.obs['roi'] = 'other'

adata.obs['roi'] = adata.obs['cell_id'].map(roi_mapping).fillna(adata.obs['roi'])

print(adata.obs.head())
print(adata.obs['roi'].value_counts())

In [ ]:
def assign_roi_type(roi_value):
    if isinstance(roi_value, str):
        if roi_value.startswith('e'):
            return 'embryo'
        elif roi_value.startswith('p'):
            return 'placenta'
        else:
            return 'other'
    else:
        return 'other'  

adata.obs['roi_type'] = adata.obs['roi'].apply(assign_roi_type)
print(adata.obs['roi_type'].value_counts())

# 3 QC

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count % : {cprobes}")
print(f"Negative decoding count % : {cwords}")

In [ ]:
## final QC
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=50)

In [ ]:
adata.write('adata_merge_after_QC_10_50.h5ad')

# 4 Pross embryo and embryo proper and extraembryonic 

In [ ]:
embryo = adata[adata.obs['roi_type'] == 'embryo']

In [ ]:

if embryo.n_obs == 0:
    print("Warning: embryo is empty. Aborting processing.")
else:
    print(f"Info: embryo selected with {embryo.n_obs} cells.")
    

    if 'counts_raw' in embryo.layers:
        embryo.X = embryo.layers['counts_raw'].copy()
        print("Info: .X initialized from 'counts_raw' layer.")
    

    sc.pp.normalize_total(embryo, inplace=True)
    sc.pp.log1p(embryo)

    sc.pp.highly_variable_genes(embryo, n_top_genes=2000)
    embryo.raw = embryo.copy()
    

    sc.pp.scale(embryo, max_value=10)

    sc.tl.pca(embryo, use_highly_variable=True)

    sc.pp.neighbors(embryo)

    sc.tl.umap(embryo, min_dist=0.01, spread=2)

    sc.tl.leiden(embryo, resolution=1, key_added='leiden_1')
    
    print(f"embryo (n={embryo.n_obs}) processing complete.")
    print(f"Total genes retained: {embryo.n_vars}")
    print(f"Highly variable genes used for analysis: {embryo.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(embryo, 
                        groupby="leiden_hvg_1", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
sc.tl.dendrogram(embryo, groupby="leiden_hvg_1", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',
sc.pl.dendrogram(embryo, groupby="leiden_hvg_1")

In [ ]:
embryo.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryo\whole_embryo_hvg_res_1_updata.h5ad")

# 5 Figure 1b

In [ ]:

tsne_coords =embryo.obsm['X_umap']

leiden_clusters = embryo.obs['leiden_hvg_1']

import pandas as pd
tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP1', 'UMAP2'])
tsne_df['leiden_hvg_1'] = leiden_clusters.values

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.scatterplot(data=tsne_df, x='UMAP1', y='UMAP2', hue='leiden_hvg_1', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\2st_2025_10_07\Figure3\umap_gut.pdf', format="pdf",dpi=900,bbox_inches='tight')
plt.show()

# 6 Figure 1d

In [ ]:
my_genes= [
    "HBZ", "SLC4A1",      
    "CSF3R", "FCGR2A",    
    "MRC1", "CD14",     
    "F13A1", "FCGBP",  
    "KDR", "PLVAP",       
    "CDH5", "TIE1",       
    "POSTN", "THBD",     
    "EGFL6", "COL4A1",    
    "PITX2", "HOXA13",     
    "GATA6", "TGFB2",    
    "IGF2", "HLX",        
    "LAPTM4B","HOXB9",           
    "DLL3", "CXCR4",      
    "CDX2", "FZD10",     
    "SOX10", "MPZ",      
    "FST", "PAX3",       
    "SFRP1",
    "TUBB2B", "SOX2",    
    "TPBG", "ZIC2",      
    "SHH", "SOX9",      
    "EPCAM", "GRHL2",   
    "BCAM", "FREM2",     
    "FOXA1", "HNF1B",    
    "DSG2", "RCN2",       
    "LDB3", "ACTN2",     
    "GABRP", "VTCN1",    
    "APOB", "PLG"        
]

In [ ]:
sc.pl.dotplot(
    embryo,
    var_names=my_genes,
    groupby='leiden_hvg_1',  
    standard_scale='var',  
    cmap="Reds",
    figsize=(14, 6),
    dendrogram=True,
    save='DEG_whole_embryo_Sub_normalize.pdf'
)